In [ ]:
from math import log
import re
import sys
import jax, numpyro
import jax.numpy as jnp
import numpy as np

def linear_model(logage, a, b):
    logv = a * logage + b
    return logv


class RotEvol():
    def __init__(self):
        pass

re = RotEvol()

def numpyro_model(re, linear_age=False):
    a = numpyro.sample("a", numpyro.distributions.Normal(-1, 1))
    b = numpyro.sample("b", numpyro.distributions.Normal(-1, 1))

    with numpyro.plate("stars",re.N):
        cosi  = numpyro.sample("cosi", numpyro.distributions.Uniform(-1, 1))
        if linear_age:
            age = numpyro.sample("age", numpyro.distributions.Uniform(0, 13.8))
            logage = numpyro.deterministic("logage", np.log10(age))
        else:
            logage = numpyro.sample("logage", numpyro.distributions.Uniform(7, 10.139879086401237))
            age = numpyro.deterministic("age", 10**logage)
        v = numpyro.deterministic("v", 10**linear_model(logage, a, b))
        vsini = numpyro.deterministic("vsini", v * jnp.sqrt(1 - cosi**2))

        numpyro.sample("obs", numpyro.distributions.Normal(vsini, re.vsini_err), obs=re.vsini_obs)


